In [ ]:
mport scipy.linalg as la
import warnings
warnings.filterwarnings('ignore')

from linearmodels.iv import IV2SLS
from statsmodels.tools import add_constant as sm_add_const

#IN_PKL = "/Users/trinityrose/Desktop/acs_ssc_msub.pkl"

df_reg = pd.read_pickle(taxsim_output2)
print(f"Loaded: {df_reg.shape}")
print(f"Married share: {df_reg['sscouple_mar'].mean():.3f}")

In [ ]:
# Cell 2 — Prepare variables
df_reg["married"]        = df_reg["sscouple_mar"].astype(float)
df_reg["msub_obs_k"]     = df_reg["total_sub_obs"]  / 1000    # was msub_total_obs
df_reg["msub_hat_k"]     = df_reg["total_sub_pred"] / 1000    # was msub_total_hat
df_reg["legal_marriage"] = df_reg["staterecog_policy"].astype(float)
df_reg["male"]         = df_reg["r_male"].astype(float)
df_reg["any_children"] = df_reg["c_anychildren"].astype(float)
df_reg["n_children"]   = df_reg["c_children"].astype(float)
df_reg["age_older"]    = df_reg["c_agemax"].astype(float)
df_reg["age_diff"]     = df_reg["c_agediff"].astype(float)
df_reg["edu_max"]      = df_reg["c_edumax"].astype(float)
df_reg["edu_diff"]     = df_reg["c_edudiff"].astype(float)
df_reg["same_race"]    = df_reg["c_racecomp"].astype(float)
df_reg["medicaid_exp"] = df_reg["medicaid_exp"].astype(float)
df_reg["earn_split_obs"] = df_reg["c_incearn_split"].astype(float)
df_reg["earn_split_hat"] = df_reg["hat_c_split"].astype(float)
df_reg["earn_obs"]  = df_reg["c_incearn"] / 10000
df_reg["earn_obs2"] = df_reg["earn_obs"] ** 2
df_reg["earn_obs3"] = df_reg["earn_obs"] ** 3
df_reg["earn_obs4"] = df_reg["earn_obs"] ** 4
df_reg["earn_obs5"] = df_reg["earn_obs"] ** 5

df_reg["earn_hat"]  = df_reg["hat_c_incearn"] / 10000
df_reg["earn_hat2"] = df_reg["earn_hat"] ** 2
df_reg["earn_hat3"] = df_reg["earn_hat"] ** 3
df_reg["earn_hat4"] = df_reg["earn_hat"] ** 4
df_reg["earn_hat5"] = df_reg["earn_hat"] ** 5

for i in range(1, 6):
    df_reg[f"hat_earn{i}"]  = df_reg[f"hat_c_incearn{i}"].astype(float)
    df_reg[f"hat_split{i}"] = df_reg[f"hat_c_split{i}"].astype(float)

df_reg["year_fe"]  = df_reg["year"].astype(str)
df_reg["state_fe"] = df_reg["statefip"].astype(str)

key_cols = ["married", "msub_obs_k", "msub_hat_k", "legal_marriage",
            "male", "any_children", "n_children", "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race", "medicaid_exp",
            "earn_split_obs", "earn_split_hat"]
df_reg = df_reg.dropna(subset=key_cols).copy()
print(f"Analysis sample: {len(df_reg):,}")
print(f"Married share:   {df_reg['married'].mean():.3f}  (paper: 0.432)")

In [ ]:
def selected_from_lasso(X: pd.DataFrame, model, prefix: str):
    """
    Return:
      - renamed dataframe of selected columns
      - list of renamed selected column names
      - raw selected column names
    """
    coef = pd.Series(model.coef_, index=X.columns)
    selected_raw = coef.index[coef != 0].tolist()

    X_sel = X[selected_raw].copy()
    renamed = {c: f"{prefix}{c}" for c in selected_raw}
    X_sel = X_sel.rename(columns=renamed)

    selected_renamed = list(X_sel.columns)
    return X_sel, selected_renamed, selected_raw

In [ ]:
Xr_sel, lasso_r_cols, lasso_r_raw = selected_from_lasso(X_r, m2_r, prefix="rL_")
Xsp_sel, lasso_sp_cols, lasso_sp_raw = selected_from_lasso(X_sp, m2_sp, prefix="spL_")

lasso_controls = lasso_r_cols + lasso_sp_cols
print(f"Selected respondent controls: {len(lasso_r_cols)}")
print(f"Selected spouse controls: {len(lasso_sp_cols)}")
print(f"Total LASSO controls: {len(lasso_controls)}")

In [ ]:
df_reg = df_reg.reset_index(drop=True).copy()
Xr_sel = Xr_sel.reset_index(drop=True)
Xsp_sel = Xsp_sel.reset_index(drop=True)

df_reg = pd.concat([df_reg, Xr_sel, Xsp_sel], axis=1)
print(df_reg.shape)

lasso_block = df_reg[lasso_controls].copy()
lasso_block = clean_design_matrix(lasso_block)

# Replace in df_reg
keep_lasso_controls = list(lasso_block.columns)
df_reg = pd.concat([df_reg.drop(columns=lasso_controls, errors="ignore"), lasso_block], axis=1)

lasso_controls = keep_lasso_controls
print(f"Usable LASSO controls after cleaning: {len(lasso_controls)}")

In [ ]:
# Baseline controls used throughout
base_controls = [
    "legal_marriage",      # = staterecog_policy
    "medicaid_exp",
    "male",                # = r_male
    "any_children",        # = c_anychildren
    "n_children",          # = c_children
    "age_older",           # = c_agemax
    "age_diff",            # = c_agediff
    "edu_max",             # = c_edumax
    "edu_diff",            # = c_edudiff
    "same_race",           # = c_racecomp
]

# OLS columns 3 and 5 use observed earnings controls
ols_income_controls = [
    "earn_obs", "earn_obs2", "earn_obs3", "earn_obs4", "earn_obs5",
    "earn_split_obs"
]

# IV columns 4 and 6 use predicted earnings controls
iv_income_controls = [
    "earn_hat", "earn_hat2", "earn_hat3", "earn_hat4", "earn_hat5",
    "earn_split_hat"
]

In [ ]:
def make_dummies(df, col):
    return pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=float)

def clean_design_matrix(X: pd.DataFrame) -> pd.DataFrame:
    X = X.loc[:, X.nunique(dropna=False) > 1]   # drop constants
    X = X.T.drop_duplicates().T                 # drop exact duplicates
    return X

def run_spec(df, spec="1", iv=False, lasso_controls=None):
    """
    spec:
      '1' = baseline
      '3' = add income polynomial + earnings split
      '4' = add income polynomial + earnings split + non-zero LASSO controls
    iv:
      False = OLS columns
      True  = IV columns
    """
    lasso_controls = lasso_controls or []

    controls = base_controls.copy()

    # Model 3 / 4 income controls
    if spec in ["3", "4"]:
        if iv:
            controls += iv_income_controls
        else:
            controls += ols_income_controls

    # Model 4 adds non-zero LASSO controls
    if spec == "4":
        controls += lasso_controls

    # Fixed effects
    year_dums = make_dummies(df, "year_fe")
    state_dums = make_dummies(df, "state_fe")

    X = pd.concat([df[controls], year_dums, state_dums], axis=1).astype(float)
    X = clean_design_matrix(X)
    X = sm_add_const(X, has_constant="add")

    y = df["married"]

    if not iv:
        # OLS columns include observed subsidy directly in X
        X.insert(1, "msub_obs_k", df["msub_obs_k"].values)
        X = clean_design_matrix(X)
        res = IV2SLS(y, X, None, None).fit(cov_type="robust")
    else:
        # IV columns instrument observed subsidy with predicted subsidy
        endog = df[["msub_obs_k"]]
        instr = df[["msub_hat_k"]]
        res = IV2SLS(y, X, endog, instr).fit(cov_type="robust")

    return res

In [ ]:
df_reg = df_reg.reset_index(drop=True).copy()
Xr_sel = Xr_sel.reset_index(drop=True)
Xsp_sel = Xsp_sel.reset_index(drop=True)

df_reg = pd.concat([df_reg, Xr_sel, Xsp_sel], axis=1)
print(df_reg.shape)

lasso_block = df_reg[lasso_controls].copy()
lasso_block = clean_design_matrix(lasso_block)

# Replace in df_reg
keep_lasso_controls = list(lasso_block.columns)
df_reg = pd.concat([df_reg.drop(columns=lasso_controls, errors="ignore"), lasso_block], axis=1)

# Keep top 20 per side
coef_r = pd.Series(m2_r.coef_, index=X_r.columns)
top_r = coef_r.abs().sort_values(ascending=False).head(20).index

coef_sp = pd.Series(m2_sp.coef_, index=X_sp.columns)
top_sp = coef_sp.abs().sort_values(ascending=False).head(20).index

lasso_controls = [f"rL_{c}" for c in top_r] + [f"spL_{c}" for c in top_sp]

print(f"Usable LASSO controls after cleaning: {len(lasso_controls)}")

In [ ]:
print("Col 1: OLS baseline...")
col1 = run_spec(df_reg, spec="1", iv=False)

print("Col 2: IV baseline...")
col2 = run_spec(df_reg, spec="1", iv=True)

print("Col 3: OLS + observed earnings polynomial + observed split...")
col3 = run_spec(df_reg, spec="3", iv=False)

print("Col 4: IV + predicted earnings polynomial + predicted split...")
col4 = run_spec(df_reg, spec="3", iv=True)

print("Col 5: OLS + observed earnings polynomial + observed split + LASSO controls...")
col5 = run_spec(df_reg, spec="4", iv=False, lasso_controls=lasso_controls)

print("Col 6: IV + predicted earnings polynomial + predicted split + LASSO controls...")
col6 = run_spec(df_reg, spec="4", iv=True, lasso_controls=lasso_controls)